In [ ]:
def write_runscript_from_config(config_data: dict, output_path: str):
    """
    根据配置字典生成一个类似 runscript_test 的固定格式运行文件。
    该版本能处理值为列表（每场景不同）或字符串（所有场景通用）的参数。

    参数:
    config_data (dict): 包含所有模拟参数的字典。
    output_path (str): 输出文件的路径。
    """
    lines = []

    # --- 1. 提取通用设置 ---
    general_config = config_data.get("SETUP_GENERAL", {})

    print(general_config)

    grid_line = ' '.join(
        map(str, general_config.get("grid_dims", [1, 1, 1, 1, 1])))
    lines.append(grid_line)

    lines.append(general_config.get("site_data_file"))
    lines.append(general_config.get("topography_data_file"))

    num_scenes = general_config.get("num_scenes", 0)
    num_runs = general_config.get("num_runs", 1)
    lines.append(f"{num_scenes} {num_runs}")

    # --- 2. 循环处理每个场景 ---
    scenes_config = config_data.get("SETUP_SCENES", {})
    # 定义场景中文件参数的顺序
    scene_file_keys = [
        "weather_data_files", "weather_options_files", "land_management_files",
        "plant_management_files", "soil_output_1", "atmospheric_output",
        "n_flux_output", "p_flux_output", "soil_output_2", "soil_output_3",
        "water_props_output", "n_props_output", "p_props_output", "t_props_output"
    ]

    for i in range(num_scenes):
        lines.append("1 1")

        for key in scene_file_keys:
            config_value = scenes_config.get(key)

            # **智能处理逻辑**
            # 如果值是列表，则按场景索引取值
            if isinstance(config_value, list):
                if i < len(config_value):
                    lines.append(config_value[i])
                else:
                    lines.append("NO_FILE_SPECIFIED_IN_LIST")
            # 如果值是字符串，则所有场景都使用该值
            elif isinstance(config_value, str):
                lines.append(config_value)
            # 如果未提供值
            else:
                lines.append("NO_FILE_SPECIFIED")

    # --- 3. 添加结束标志 ---
    lines.append("0 0")

    # --- 4. 将所有行写入文件 ---
    try:
        with open(output_path, 'w') as f:
            f.write('\n'.join(lines))
        print(f"文件已成功生成在: {output_path}")
    except IOError as e:
        print(f"写入文件时出错: {e}")


# =======================================================================
#                           --- 使用示例 ---
# =======================================================================

if __name__ == '__main__':
    # 1. 定义新的配置字典，注意值的变化
    my_config = {
        "SETUP_GENERAL": {
            "grid_dims": [1, 1, 1, 1, 1],
            "site_data_file": "st022852.txt",
            "topography_data_file": "tp022852.txt",
            "num_scenes": 3,
            "num_runs": 1
        },
        "SETUP_SCENES": {
            # 这个参数保持为列表，因为每个场景的气象数据通常不同
            "weather_data_files":     ['w1980022852', 'w1981022852', 'w1982022852'],
            # --- 以下参数已从列表改为单个字符串 ---
            "weather_options_files":  ['opt1800', 'opt1801', 'opt1802'],
            "land_management_files":  'NO',
            "plant_management_files": 'pft_arctic_g',
            "soil_output_1":          'NO',
            "atmospheric_output":     'NO',
            "n_flux_output":          'NO',
            "p_flux_output":          'NO',
            "soil_output_2":          'ds',
            "soil_output_3":          'dc',
            "water_props_output":     'dw',
            "n_props_output":         'dn',
            "p_props_output":         'NO',
            "t_props_output":         'NO'
        }
    }

    # 2. 调用函数，指定输出文件名
    output_filename = "runscript_generated_single_element.txt"
    write_runscript_from_config(my_config, output_filename)

In [ ]:
import collections.abc


def write_sitedata_from_config(config_data: dict, output_path: str):
    """
    根据配置字典生成一个 site_data 格式的文件 (如 st022852)。

    参数:
    config_data (dict): 包含所有站点参数的字典。
    output_path (str): 输出文件的路径。
    """
    lines = []
    params = config_data.get("SITE_PARAMETERS", {})

    # --- Line 1: 地理和基本水文信息 ---
    line1 = (f"{params.get('latitude', 0.0)} {params.get('altitude', 0.0)} "
             f"{params.get('mean_temp_c', 0.0)} {params.get('water_table_flag', 0.0)}")
    lines.append(line1)

    # --- Line 2: 大气组分 ---
    # O2, N2, CO2, CH4, N2O, NH3
    atm_comp = params.get('atm_composition_ppm', [])
    lines.append(' '.join(map(str, atm_comp)))

    # --- Line 3: 气候、网格和水文参数 ---
    # 柯本气候区, 盐分, 侵蚀选项(0=冻融), 网格连接(1=行),自然/人工地下水深, 地下水坡度
    #
    cgh_params = params.get('climate_grid_hydro_params', [])
    lines.append(' '.join(map(str, cgh_params)))

    # --- Line 4: 边界条件 (合并多个参数) ---
    bc_surf = params.get('bc_surface_runoff_nesw', [
                         0.0]*4)  # N E S W 边界条件 (地表径流)
    bc_sub = params.get('bc_subsurface_flow_nesw', [
                        0.0]*4)  # N E S W 边界条件 (地下径流)
    dist_wt = params.get('dist_water_table_nesw', [
                         0.0]*4)  # N E S W 到地下水表的距离 (m)
    lower_bc = params.get('lower_bc_water_flow', 0.0)

    # 确保 lower_bc 是一个可迭代对象以便拼接
    if not isinstance(lower_bc, collections.abc.Iterable):
        lower_bc = [lower_bc]

    full_bc_line_values = list(bc_surf) + list(bc_sub) + \
        list(dist_wt) + list(lower_bc)
    lines.append(' '.join(map(str, full_bc_line_values)))

    # --- Line 5: 东西向宽度 ---
    lines.append(str(params.get('width_we_column', 1.0)))

    # --- Line 6: 南北向宽度 ---
    lines.append(str(params.get('width_ns_row', 1.0)))

    # --- 写入文件 ---
    try:
        with open(output_path, 'w') as f:
            f.write('\n'.join(lines))
        print(f"文件已成功生成在: {output_path}")
    except IOError as e:
        print(f"写入文件时出错: {e}")

# =======================================================================
#                           --- 使用示例 ---
# =======================================================================


if __name__ == '__main__':
    # 1. 定义与 Namelist 结构一致的Python字典
    my_site_config = {
        "SITE_PARAMETERS": {
            "latitude": 69.1,
            "altitude": 130.0,
            "mean_temp_c": -8.0,
            "water_table_flag": 1.0,  # 地下水表标记 (1 = 自然静止)
            # O2, N2, CO2, CH4, N2O, NH3
            "atm_composition_ppm": [210000.0, 780000.0, 282.9, 1.8, 0.3, 0.005],
            # 柯本气候区, 盐分, 侵蚀选项(0=冻融), 网格连接(1=行),自然地下水深, 人工地下水深, 地下水坡度
            "climate_grid_hydro_params": [62, 0, -1, 1, 3, 100.0, 0.0],
            # N E S W 边界条件 (地表径流)
            "bc_surface_runoff_nesw": [0.0, 0.0, 0.0, 0.0],
            # N E S W 边界条件 (地下径流)
            "bc_subsurface_flow_nesw": [0.0, 0.0, 0.0, 0.0],
            # N E S W 到地下水表的距离 (m)
            "dist_water_table_nesw": [0.0, 0.0, 0.0, 0.0],
            "lower_bc_water_flow": 0.0,  # 水流的下边界条件
            "width_we_column": 1.0,
            "width_ns_row": 1.0
        }
    }

    # 2. 调用函数，指定输出文件名
    output_filename = "st022852.txt"
    write_sitedata_from_config(my_site_config, output_filename)

In [ ]:
def write_topography_from_config(config_data: dict, output_path: str):
    """
    根据配置字典生成一个 topography_data 格式的文件 (如 tp022852)。

    参数:
    config_data (dict): 包含所有地形参数的字典。
    output_path (str): 输出文件的路径。
    """
    lines = []
    params = config_data.get("TOPOGRAPHY_PARAMETERS", {})

    # --- 构造第一行 ---
    grid_struct = params.get('inner_grid_structure', [1, 1, 1, 1])
    aspect = params.get('landscape_aspect_deg', 0.0)
    slope_ew = params.get('slope_ew_deg', 0.0)
    slope_ns = params.get('slope_ns_deg', 0.0)
    placeholder = params.get('placeholder_value', 0.0)

    # 将所有数值合并到一个列表中
    line1_values = list(grid_struct) + \
        [aspect, slope_ew, slope_ns, placeholder]

    # 将列表转换为一个用空格分隔的字符串
    lines.append(' '.join(map(str, line1_values)))

    # --- 构造第二行 ---
    soil_file = params.get('soil_data_file', 'default_soil.txt')
    lines.append(soil_file)

    # --- 写入文件 ---
    try:
        with open(output_path, 'w') as f:
            f.write('\n'.join(lines))
        print(f"文件已成功生成在: {output_path}")
    except IOError as e:
        print(f"写入文件时出错: {e}")

# =======================================================================
#                           --- 使用示例 ---
# =======================================================================


if __name__ == '__main__':
    # 1. 定义与 Namelist 结构一致的Python字典
    my_topo_config = {
        "TOPOGRAPHY_PARAMETERS": {
            "inner_grid_structure": [1, 1, 1, 1],
            "landscape_aspect_deg": 90.0,
            "slope_ew_deg": 1.0,
            "slope_ns_deg": 0.01,
            "placeholder_value": 0.0,
            "soil_data_file": "s022852.txt"
        }
    }

    # 2. 调用函数，指定输出文件名
    output_filename = "tp022852.txt"
    write_topography_from_config(my_topo_config, output_filename)

In [ ]:
def write_soildata_from_config(config_data: dict, output_path: str):
    """
    根据配置字典生成一个 soil_data 格式的文件 (如 s022852)。
    函数内部通过固定的key顺序列表来确保输出格式的准确性。

    参数:
    config_data (dict): 包含所有土壤参数的字典。
    output_path (str): 输出文件的路径。
    """
    lines = []
    globals_params = config_data.get("SOIL_GLOBALS", {})
    layers_params = config_data.get("SOIL_LAYERS", {})

    # --- 1. 构造全局参数行 (第一行) ---
    # 定义第一行参数的准确顺序
    global_keys_in_order = [
        'water_potential_fc_mpa', 'water_potential_wp_mpa', 'wet_soil_albedo',
        'litter_ph', 'litter_fine_c', 'litter_fine_n', 'litter_fine_p',
        'litter_woody_c', 'litter_woody_n', 'litter_woody_p', 'litter_manure_c',
        'litter_manure_n', 'litter_manure_p', 'litter_type_plant',
        'litter_type_manure', 'num_surface_layers', 'num_max_rooting_layers',
        'num_additional_layers_w_data', 'num_additional_layers_wo_data',
        'profile_type'
    ]

    global_values = [globals_params.get(key, 0.0)
                     for key in global_keys_in_order]
    # 第一行使用逗号分隔
    lines.append(','.join(map(str, global_values)))

    # --- 2. 构造分层参数行 ---
    # 定义所有分层参数的准确顺序
    layer_keys_in_order = [
        'layer_depth_bottom_m', 'bulk_density_mg_m3', 'field_capacity_m3_m3',
        'wilting_point_m3_m3', 'vertical_ksat_mm_h', 'lateral_ksat_mm_h',
        'sand_contents_kg_mg', 'silt_contents_kg_mg', 'macropore', 'rock_fraction',
        'ph', 'cation_exchange_capacity', 'anion_exchange_capacity',
        'total_soc_kg_mg', 'poc_kg_mg', 'son_g_mg', 'sop_g_mg',
        'soluble_exch_nh4_g_mg', 'soluble_exch_no3_g_mg', 'soluble_exch_h2po4_g_mg',
        'soluble_al_g_mg', 'soluble_fe_g_mg', 'soluble_ca_g_mg', 'soluble_mg_g_mg',
        'soluble_na_g_mg', 'soluble_k_g_mg', 'soluble_so4s_g_mg', 'soluble_cl_g_mg',
        'alpo4_mineral_g_mg', 'fepo4_mineral_g_mg', 'cahpo4_mineral_g_mg',
        'apatite_mineral_g_mg', 'aloh3_mineral_g_mg', 'feoh3_mineral_g_mg',
        'caso4_mineral_g_mg', 'caco3_mineral_g_mg', 'gapon_ca_nh4', 'gapon_ca_h',
        'gapon_ca_al', 'gapon_ca_mg', 'gapon_ca_na', 'gapon_ca_k',
        'initial_water_contents', 'initial_ice_contents', 'initial_c_fine_litter',
        'initial_n_fine_litter', 'initial_p_fine_litter', 'initial_c_woody_litter',
        'initial_n_woody_litter', 'initial_p_woody_litter',
        'initial_c_manure_litter', 'initial_n_manure_litter', 'initial_p_manure_litter'
    ]

    for key in layer_keys_in_order:
        layer_values = layers_params.get(key, [])
        # 后续行同样使用逗号分隔
        lines.append(','.join(map(str, layer_values)))

    # --- 3. 写入文件 ---
    try:
        with open(output_path, 'w') as f:
            f.write('\n'.join(lines))
        print(f"文件已成功生成在: {output_path}")
    except IOError as e:
        print(f"写入文件时出错: {e}")


# =======================================================================
#                           --- 使用示例 ---
# =======================================================================
if __name__ == '__main__':
    # 定义与 Namelist 结构一致的Python字典
    # 这是 s022852 文件的完整 Python 表示
    my_soil_config = {
        "SOIL_GLOBALS": {
            'water_potential_fc_mpa': -0.03, 'water_potential_wp_mpa': -1.5,
            'wet_soil_albedo': 0.12, 'litter_ph': 3.72, 'litter_fine_c': 500.0,
            'litter_fine_n': 12.5, 'litter_fine_p': 1.25, 'litter_woody_c': 0.0,
            'litter_woody_n': 0.0, 'litter_woody_p': 0.0, 'litter_manure_c': 0.0,
            'litter_manure_n': 0.0, 'litter_manure_p': 0.0, 'litter_type_plant': 10.0,
            'litter_type_manure': 0.0, 'num_surface_layers': 1.0,
            'num_max_rooting_layers': 8.0, 'num_additional_layers_w_data': 0.0,
            'num_additional_layers_wo_data': 0.0, 'profile_type': 1.0
        },
        "SOIL_LAYERS": {
            'layer_depth_bottom_m': [0.01, 0.05, 0.15, 0.3, 0.5, 0.7, 1.16, 1.52, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'bulk_density_mg_m3': [1.1, 1.1, 1.21, 1.21, 1.47, 1.47, 1.47, 1.47, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'field_capacity_m3_m3': [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0],
            'wilting_point_m3_m3': [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0],
            'vertical_ksat_mm_h': [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0],
            'lateral_ksat_mm_h': [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0],
            'sand_contents_kg_mg': [319.71, 319.71, 319.71, 319.71, 330.14, 330.14, 330.14, 330.14, 180.0, 180.0, 180.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'silt_contents_kg_mg': [545.39, 545.39, 545.39, 545.39, 184.49, 184.49, 184.49, 184.49, 20.0, 20.0, 20.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'macropore': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'rock_fraction': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'ph': [5.77, 5.77, 5.77, 5.77, 5.8, 5.8, 5.8, 5.8, 4.35, 4.35, 4.35, 4.35, 4.35, 4.35, 4.35, 0.0, 0.0, 0.0, 0.0],
            'cation_exchange_capacity': [10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 5.0, 5.0, 5.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'anion_exchange_capacity': [3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'total_soc_kg_mg': [3.87, 2.87, 2.6, 2.6, 2.0, 2.0, 2.0, 2.0, 3.0, 0.5, 0.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'poc_kg_mg': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'son_g_mg': [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0],
            'sop_g_mg': [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0],
            'soluble_exch_nh4_g_mg': [3.0, 3.0, 3.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'soluble_exch_no3_g_mg': [12.0, 12.0, 12.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'soluble_exch_h2po4_g_mg': [10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'soluble_al_g_mg': [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'soluble_fe_g_mg': [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'soluble_ca_g_mg': [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0],
            'soluble_mg_g_mg': [0.0, 0.0, 0.0, 18.0, 18.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'soluble_na_g_mg': [0.07, 0.07, 0.07, 0.05, 0.05, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'soluble_k_g_mg': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'soluble_so4s_g_mg': [48.0, 48.0, 48.0, 48.0, 48.0, 48.0, 48.0, 48.0, 48.0, 48.0, 48.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'soluble_cl_g_mg': [35.0, 35.0, 35.0, 35.0, 35.0, 35.0, 35.0, 35.0, 35.0, 35.0, 35.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'alpo4_mineral_g_mg': [50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'fepo4_mineral_g_mg': [50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'cahpo4_mineral_g_mg': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'apatite_mineral_g_mg': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'aloh3_mineral_g_mg': [1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'feoh3_mineral_g_mg': [1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'caso4_mineral_g_mg': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'caco3_mineral_g_mg': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'gapon_ca_nh4': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
            'gapon_ca_h': [0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'gapon_ca_al': [0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'gapon_ca_mg': [0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'gapon_ca_na': [0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'gapon_ca_k': [3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'initial_water_contents': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'initial_ice_contents': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'initial_c_fine_litter': [45.0, 60.0, 75.0, 75.0, 60.0, 60.0, 45.0, 45.0, 15.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'initial_n_fine_litter': [1.5, 2.0, 2.5, 2.5, 2.0, 2.0, 1.5, 1.5, 0.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'initial_p_fine_litter': [0.15, 0.2, 0.25, 0.25, 0.2, 0.2, 0.15, 0.15, 0.05, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'initial_c_woody_litter': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'initial_n_woody_litter': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'initial_p_woody_litter': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'initial_c_manure_litter': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'initial_n_manure_litter': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            'initial_p_manure_litter': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
        }
    }

    # 调用函数，指定输出文件名
    output_filename = "s022852.txt"
    write_soildata_from_config(my_soil_config, output_filename)

In [ ]:
def write_crop_params_from_config(config_data: dict, output_path: str):
    """
    根据配置字典生成一个作物/植被参数格式的文件 (如 ndlf62)。
    函数通过一个固定的键序列表来确保输出格式的准确性。

    参数:
    config_data (dict): 包含所有作物参数的字典。
    output_path (str): 输出文件的路径。
    """
    lines = []
    params = config_data.get("CROP_PARAMETERS", {})

    # 定义参数在输出文件中的准确顺序
    keys_in_order = [
        'biology_and_phenology', 'photosynthesis_biochem', 'leaf_optical_props',
        'development_and_temp', 'flowering_and_photoperiod', 'organ_growth',
        'canopy_structure', 'seed_and_establishment', 'root_properties',
        'nh4_uptake_kinetics', 'no3_uptake_kinetics', 'h2po4_uptake_kinetics',
        'water_relations', 'organ_growth_yield', 'organ_nc_ratio',
        'organ_pc_ratio'
    ]

    for key in keys_in_order:
        values = params.get(key, [])
        # 将列表中的所有元素转换为字符串，然后用空格连接
        # 使用格式化来处理科学记数法，使其输出更美观
        formatted_values = []
        for v in values:
            if isinstance(v, float) and (v < 1e-3 or v > 1e4) and v != 0.0:
                formatted_values.append(f"{v:.1E}".replace(
                    "E-0", "E-").replace("E+0", "E+"))
            else:
                formatted_values.append(str(v))
        lines.append(' '.join(formatted_values))

    # 写入文件
    try:
        with open(output_path, 'w') as f:
            f.write('\n'.join(lines))
        print(f"文件已成功生成在: {output_path}")
    except IOError as e:
        print(f"写入文件时出错: {e}")


# =======================================================================
#                           --- 使用示例 ---
# =======================================================================
if __name__ == '__main__':
    # 定义与 Namelist 结构一致的Python字典
    # 这是 ndlf62 文件的完整 Python 表示
    my_crop_config = {
        "CROP_PARAMETERS": {
            'biology_and_phenology': [3, 2, 1, 1, 0, 1, 2, 2, 0, 2, 1.00],
            'photosynthesis_biochem': [45.0, 9.5, 0.0, 12.5, 500.0, 0.0, 0.125, 0.0, 405.0, 0.025, 0.0, 0.70],
            'leaf_optical_props': [0.150, 0.075, 0.150, 0.075],
            'development_and_temp': [0.015, 0.009, -10.0, 60.0, 720.0, 2.5, 0.10],
            'flowering_and_photoperiod': [6.5, 2.5, -1.0, 0.5],
            'organ_growth': [0.00333, 0.0125, 0.015],
            'canopy_structure': [0.25, 0.25, 0.25, 0.25, 0.475, 90.0, 0.0],
            'seed_and_establishment': [1.0, 1.0, 0.10, 1.0, 2.0E-04, 100.0],
            'root_properties': [2.5E-04, 1.0E-04, 0.05, 0.10, 1.0E+04, 4.0E+10, 1.0E-02, 250.0, 250.0],
            'nh4_uptake_kinetics': [5.0E-03, 0.40, 0.0125],
            'no3_uptake_kinetics': [5.0E-03, 0.35, 0.030],
            'h2po4_uptake_kinetics': [1.0E-03, 0.075, 0.002],
            'water_relations': [-1.25, -5.0, 2.5E+03],
            'organ_growth_yield': [7.6E-01, 7.6E-01, 8.0E-01, 8.8E-01, 7.6E-01, 7.6E-01, 8.8E-01, 7.6E-01, 7.2E-01],
            'organ_nc_ratio': [4.0E-02, 2.0E-02, 0.40E-02, 2.0E-02, 2.0E-02, 2.0E-02, 4.0E-02, 2.0E-02, 10.0E-02],
            'organ_pc_ratio': [4.0E-03, 2.0E-03, 0.40E-03, 2.0E-03, 2.0E-03, 2.0E-03, 4.0E-03, 2.0E-03, 10.0E-03]
        }
    }

    moss62_config = {
        "CROP_PARAMETERS": {
            'biology_and_phenology': [3, 0, 1, 1, 3, 1, 2, 2, 0, 2, 0],
            'photosynthesis_biochem': [45.0, 9.5, 0.0, 12.5, 500.0, 0.0, 0.125, 0.0, 405.0, 0.025, 0.0, 0.70],
            'leaf_optical_props': [0.150, 0.075, 0.150, 0.075],
            'development_and_temp': [0.015, 0.009, -10.0, 60.0, 450.0, 1.0, 1.0],
            'flowering_and_photoperiod': [6.5, 2.5, -1.0, 0.5],
            'organ_growth': [0.00167, 0.0125, 0.015],
            'canopy_structure': [0.25, 0.25, 0.25, 0.25, 1.00, 90.0, 0.00],
            'seed_and_establishment': [10.0, 10.0, 1.0E-03, 1.0E-03, 2.5E-06, 0.0],
            'root_properties': [1.0E-04, 5.0E-06, 0.00, 0.10, 1.0E+04, 4.0E+09, 1.0E-02, 250.0, 250.0],
            'nh4_uptake_kinetics': [5.0E-04, 0.40, 0.0125],
            'no3_uptake_kinetics': [5.0E-04, 0.35, 0.030],
            'h2po4_uptake_kinetics': [1.0E-03, 0.075, 0.002],
            'water_relations': [-1.25, 0.0, 1.50E+02],
            'organ_growth_yield': [7.6E-01, 7.6E-01, 8.0E-01, 8.8E-01, 7.6E-01, 7.6E-01, 8.8E-01, 7.6E-01, 7.2E-01],
            'organ_nc_ratio': [4.0E-02, 2.0E-02, 1.0E-02, 2.0E-02, 2.0E-02, 2.0E-02, 4.0E-02, 2.0E-02, 10.0E-02],
            'organ_pc_ratio': [4.0E-03, 2.0E-03, 1.0E-03, 2.0E-03, 2.0E-03, 2.0E-03, 4.0E-03, 2.0E-03, 10.0E-03]
        }
    }

    # 调用函数，指定输出文件名
    output_filename = "moss62"
    write_crop_params_from_config(moss62_config, output_filename)

In [ ]:
def write_weather_options_from_config(config_data: dict, output_path: str):
    """
    根据配置字典生成一个 weather_options 格式的文件 (如 opt1800)。
    函数通过一个固定的键序列表来确保输出格式的准确性。

    参数:
    config_data (dict): 包含所有天气选项参数的字典。
    output_path (str): 输出文件的路径。
    """
    lines = []
    params = config_data.get("WEATHER_OPTIONS", {})

    # --- 1. 构造文件内容 ---
    # 定义参数在输出文件中的准确顺序
    keys_in_order = [
        'scenario_start_date', 'scenario_end_date', 'run_start_date',
        'generate_files_data', 'generate_checkpoint', 'resume_from_earlier',
        'annual_change_params_1', 'annual_change_params_2', 'annual_change_params_3', 'annual_change_params_4',
        'calc_and_output_freq'
    ]
    
    for key in keys_in_order:
        value = params.get(key)
        if isinstance(value, list):
            # 将列表中的所有元素转换为字符串，然后用逗号连接
            lines.append(','.join(map(str, value)))
        elif value is not None:
            # 直接添加字符串值
            lines.append(str(value))
        else:
            # 如果某个键不存在，添加一个空行或错误标记
            lines.append("") 

    # --- 2. 写入文件 ---
    try:
        with open(output_path, 'w') as f:
            f.write('\n'.join(lines))
        print(f"文件已成功生成在: {output_path}")
    except IOError as e:
        print(f"写入文件时出错: {e}")


# =======================================================================
#                           --- 使用示例 ---
# =======================================================================
if __name__ == '__main__':
    # --- opt1800 的 Python 字典表示 ---
    opt1800_config = {
        "WEATHER_OPTIONS": {
            'scenario_start_date': "01011800",
            'scenario_end_date': "31121800",
            'run_start_date': "01011800",
            'generate_files_data': "NO",
            'generate_checkpoint': "NO",
            'resume_from_earlier': "NO",
            'annual_change_params_1': [1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0018, 1, 1],
            'annual_change_params_2': [1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0018, 1, 1],
            'annual_change_params_3': [1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0018, 1, 1],
            'annual_change_params_4': [1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0018, 1, 1],
            'calc_and_output_freq': [30, 20, 3, 1, -1, 2]
        }
    }

    write_weather_options_from_config(opt1800_config, "opt1800")
    
    # --- opt1801 的 Python 字典表示 ---

    opt1801_config = opt1800_config.copy()
    opt1801_config['WEATHER_OPTIONS'].update({'scenario_start_date': "01011801",
            'scenario_end_date': "31121801"})

    # 调用函数，生成两个文件
    write_weather_options_from_config(opt1801_config, "opt1801")

    # --- opt1802 的 Python 字典表示 ---

    opt1802_config = opt1800_config.copy()
    opt1802_config['WEATHER_OPTIONS'].update({'scenario_start_date': "01011802",
            'scenario_end_date': "31121802"})

    # 调用函数，生成两个文件
    write_weather_options_from_config(opt1802_config, "opt1802")
    